In [1]:
import os
import pandas as pd
import mdtraj
import time 
import copy
import numpy as np
from mdtraj.formats import DCDTrajectoryFile
from mdtraj import element
# os.chdir("D:\PROTEIN-FOLDING-DATA\Chignolin\DESRES-Trajectory_CLN025-0-all\CLN025-0-all")
os.chdir("D:\PROTEIN-FOLDING-DATA\Chignolin\DESRES-Trajectory_CLN025-0-protein\CLN025-0-protein")

In [2]:
def fix_topology(topology):

    new_top = topology.copy()

    residues = {}
    for chain in new_top.chains:
        #print chain
        for residue in chain.residues:
            resname = str(residue)
            if resname in list(residues.keys()):
                residues[resname].append(residue)
            else:
                residues[resname] = [residue]

    for resname in list(residues.keys()):
        fragments = residues[resname]
        if len(fragments) > 1:
            main_fragment = fragments[0]
            new_atom_list = []
            new_atom_list += main_fragment._atoms
            for i in range(1,len(fragments)):
                fragment = fragments[i]
                for atom in fragment.atoms:
                    atom.residue = main_fragment
                new_atom_list += fragment._atoms
                fragment._atoms = []
                fragment.chain = main_fragment.chain
            main_fragment._atoms = new_atom_list

    return new_top

def fix_traj(traj):
    time0 = time.time()
    new_traj = copy.deepcopy(traj)
    topology = new_traj.topology 

    new_top = fix_topology(topology)
    topology = new_top
    new_traj.topology = new_top 

    new_atom_sequence = [a for a in topology.atoms]
    new_index_sequence = [a.index for a in topology.atoms]

    for i in range(0, np.shape(traj.xyz)[0]):
        new_traj.xyz[i] = new_traj.xyz[i][new_index_sequence]

    for i in range(0, len(new_index_sequence)):
        new_atom_sequence[i].index = i

    time1 = time.time()
    print(time1 - time0)
    return new_traj

In [3]:
trajectory = mdtraj.load_hdf5("C:/Users/ttunj/Downloads/mlcgmd/TINGnn/src/new_traj_longer.h5")
print(f"shape:{trajectory.xyz.shape}")
print(f"10 rgs:{mdtraj.compute_rg(trajectory)[:10]}")
new_traj = fix_traj(trajectory)
new_traj = new_traj.center_coordinates(mass_weighted=False)
new_traj.superpose(new_traj[0])
new_traj.xyz = new_traj.xyz/10

shape:(9970, 166, 3)
10 rgs:[6.06458807 6.15302827 6.09471651 6.08870333 6.07470748 6.10844403
 6.19379809 6.04074732 6.06584271 5.99999019]
0.1613295078277588


In [4]:
real_trajectory = mdtraj.load_dcd("CLN025-0-protein-048.dcd", top="Topology.pdb")
print(f"shape:{real_trajectory.xyz.shape}")
print(f"10 rgs:{mdtraj.compute_rg(real_trajectory)[:10]}")
real_new_traj = fix_traj(real_trajectory)
real_new_traj = real_new_traj.center_coordinates(mass_weighted=False)
real_new_traj.superpose(real_new_traj[0])

shape:(10000, 166, 3)
10 rgs:[0.58288702 0.60559201 0.58670295 0.59660842 0.57716661 0.6162106
 0.59463657 0.61322513 0.621163   0.62112497]
0.15900325775146484


<mdtraj.Trajectory with 10000 frames, 166 atoms, 10 residues, and unitcells at 0x1ec4d79ceb0>

In [5]:
import nglview as nv
new = nv.show_mdtraj(new_traj)
# new = nv.show_mdtraj(traj)
# new = nv.show_mdtraj(trajectory)
new.clear()
new.add_ball_and_stick('protein', opacity=0.5)
# new.add_surface('protein', opacity=0.05, color='residueindex')
new

NGLWidget(max_frame=9969)

In [ ]:
import nglview as nv
real_new = nv.show_mdtraj(real_new_traj)
# new = nv.show_mdtraj(traj)
# new = nv.show_mdtraj(trajectory)
real_new.clear()
real_new.add_ball_and_stick('protein', opacity=0.5)
# new.add_surface('protein', opacity=0.05, color='residueindex')
real_new

In [10]:
top = new_traj.topology
table, bonds = top.to_dataframe()

table['radius'] = 0
table['mass'] = 0
for i in range(table.shape[0]):
    table['mass'].iloc[i] = element.get_by_symbol(table.element.iloc[i]).mass
    table['radius'].iloc[i] = element.get_by_symbol(table.element.iloc[i]).radius

C:\Users\ttunj\AppData\Local\Temp\ipykernel_20336\4041283639.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  table['mass'].iloc[i] = element.get_by_symbol(table.element.iloc[i]).mass
C:\Users\ttunj\AppData\Local\Temp\ipykernel_20336\4041283639.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  table['radius'].iloc[i] = element.get_by_symbol(table.element.iloc[i]).radius


In [11]:
table.to_json()

'{"serial":{"0":null,"1":null,"2":null,"3":null,"4":null,"5":null,"6":null,"7":null,"8":null,"9":null,"10":null,"11":null,"12":null,"13":null,"14":null,"15":null,"16":null,"17":null,"18":null,"19":null,"20":null,"21":null,"22":null,"23":null,"24":null,"25":null,"26":null,"27":null,"28":null,"29":null,"30":null,"31":null,"32":null,"33":null,"34":null,"35":null,"36":null,"37":null,"38":null,"39":null,"40":null,"41":null,"42":null,"43":null,"44":null,"45":null,"46":null,"47":null,"48":null,"49":null,"50":null,"51":null,"52":null,"53":null,"54":null,"55":null,"56":null,"57":null,"58":null,"59":null,"60":null,"61":null,"62":null,"63":null,"64":null,"65":null,"66":null,"67":null,"68":null,"69":null,"70":null,"71":null,"72":null,"73":null,"74":null,"75":null,"76":null,"77":null,"78":null,"79":null,"80":null,"81":null,"82":null,"83":null,"84":null,"85":null,"86":null,"87":null,"88":null,"89":null,"90":null,"91":null,"92":null,"93":null,"94":null,"95":null,"96":null,"97":null,"98":null,"99":nul

In [12]:
topology_object = new_traj.topology

In [13]:
# if not isinstance(topology_object, Topology):
#     topology_object = Topology.from_openmm(topology_object)

try:
    topology_dict = {
        'chains': [],
        'bonds': []
    }

    for chain in topology_object.chains:
        chain_dict = {
            'residues': [],
            'index': int(chain.index)
        }
        for residue in chain.residues:
            residue_dict = {
                'index': int(residue.index),
                'name': str(residue.name),
                'atoms': [],
                "resSeq": int(residue.resSeq),
                "segmentID": str(residue.segment_id)
            }

            for atom in residue.atoms:

                try:
                    element_symbol_string = str(atom.element.symbol)
                except AttributeError:
                    element_symbol_string = ""

                residue_dict['atoms'].append({
                    'index': int(atom.index),
                    'name': str(atom.name),
                    'element': element_symbol_string
                })
            chain_dict['residues'].append(residue_dict)
        topology_dict['chains'].append(chain_dict)

    for atom1, atom2 in topology_object.bonds:
        topology_dict['bonds'].append([
            int(atom1.index),
            int(atom2.index)
        ])

except AttributeError as e:
    raise AttributeError('topology_object fails to implement the'
        'chains() -> residue() -> atoms() and bond() protocol. '
        'Specifically, we encountered the following %s' % e)

# actually set the tables
# try:
#     self._remove_node(where='/', name='topology')
# except self.tables.NoSuchNodeError:
#     pass

# data = json.dumps(topology_dict)
# if not isinstance(data, bytes):
#     data = data.encode('ascii')

# if self.tables.__version__ >= '3.0.0':
#     self._handle.create_array(where='/', name='topology', obj=[data])
# else:
#     self._handle.createArray(where='/', name='topology', object=[data])

In [14]:
topology_dict

{'chains': [{'residues': [{'index': 0,
     'name': 'TYR',
     'atoms': [{'index': 0, 'name': 'N', 'element': 'N'},
      {'index': 1, 'name': 'H', 'element': 'H'},
      {'index': 2, 'name': 'H2', 'element': 'H'},
      {'index': 3, 'name': 'H3', 'element': 'H'},
      {'index': 4, 'name': 'CA', 'element': 'C'},
      {'index': 5, 'name': 'HA', 'element': 'H'},
      {'index': 6, 'name': 'CB', 'element': 'C'},
      {'index': 7, 'name': 'HB3', 'element': 'H'},
      {'index': 8, 'name': 'HB2', 'element': 'H'},
      {'index': 9, 'name': 'CG', 'element': 'C'},
      {'index': 10, 'name': 'CD1', 'element': 'C'},
      {'index': 11, 'name': 'HD1', 'element': 'H'},
      {'index': 12, 'name': 'CE1', 'element': 'C'},
      {'index': 13, 'name': 'HE1', 'element': 'H'},
      {'index': 14, 'name': 'CZ', 'element': 'C'},
      {'index': 15, 'name': 'OH', 'element': 'O'},
      {'index': 16, 'name': 'HH', 'element': 'H'},
      {'index': 17, 'name': 'CD2', 'element': 'C'},
      {'index': 18,

In [148]:
trajectory[0].xyz*10

array([[[-9.2591152e+00,  2.2499788e+00, -3.8949323e+00],
        [-8.8362913e+00,  1.3371778e+00, -4.1587758e+00],
        [-8.9228315e+00,  2.9659009e+00, -4.5701861e+00],
        [-1.0249209e+01,  2.1142960e+00, -4.1828609e+00],
        [-8.9245691e+00,  2.7901185e+00, -2.6225793e+00],
        [-9.6982145e+00,  3.4724486e+00, -2.3027384e+00],
        [-8.8806152e+00,  1.5544724e+00, -1.6347766e+00],
        [-9.8065586e+00,  9.6194744e-01, -1.7955983e+00],
        [-8.0323048e+00,  8.8766336e-01, -1.8994546e+00],
        [-8.8439703e+00,  1.9393897e+00, -1.9835174e-01],
        [-9.7859325e+00,  2.8576326e+00,  4.0370107e-01],
        [-1.0550913e+01,  3.2483721e+00, -2.5092125e-01],
        [-9.8002443e+00,  3.0235672e+00,  1.7963219e+00],
        [-1.0599599e+01,  3.5499632e+00,  2.2966642e+00],
        [-8.9627724e+00,  2.2921360e+00,  2.6502461e+00],
        [-9.0322723e+00,  2.4287713e+00,  4.0787048e+00],
        [-8.7361526e+00,  1.6115832e+00,  4.4862790e+00],
        [-8.00

In [25]:
np.array(table.mass)

array([14.00672 ,  1.007947,  1.007947,  1.007947, 12.01078 ,  1.007947,
       12.01078 ,  1.007947,  1.007947, 12.01078 , 12.01078 ,  1.007947,
       12.01078 ,  1.007947, 12.01078 , 15.99943 ,  1.007947, 12.01078 ,
        1.007947, 12.01078 ,  1.007947, 12.01078 , 15.99943 , 14.00672 ,
        1.007947, 12.01078 ,  1.007947, 12.01078 ,  1.007947,  1.007947,
       12.01078 , 12.01078 ,  1.007947, 12.01078 ,  1.007947, 12.01078 ,
       15.99943 ,  1.007947, 12.01078 ,  1.007947, 12.01078 ,  1.007947,
       12.01078 , 15.99943 , 14.00672 ,  1.007947, 12.01078 ,  1.007947,
       12.01078 ,  1.007947,  1.007947, 12.01078 , 15.99943 , 15.99943 ,
       12.01078 , 15.99943 , 14.00672 , 12.01078 ,  1.007947,  1.007947,
       12.01078 ,  1.007947, 12.01078 ,  1.007947,  1.007947, 12.01078 ,
        1.007947,  1.007947, 12.01078 , 15.99943 , 14.00672 ,  1.007947,
       12.01078 ,  1.007947, 12.01078 ,  1.007947,  1.007947, 12.01078 ,
        1.007947,  1.007947, 12.01078 , 15.99943 , 

In [26]:
rgs = mdtraj.compute_rg(trajectory, masses=np.array(table.mass))*10

In [28]:
rgs_wo = mdtraj.compute_rg(trajectory)*10
rgs_wo

array([6.86907686, 6.7181722 , 6.82201378, ..., 6.00798988, 6.26499992,
       5.94087869])

In [149]:
print(dir(trajectory))

['__add__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__slotnames__', '__str__', '__subclasshook__', '__weakref__', '_check_valid_unitcell', '_distance_unit', '_have_unitcell', '_rmsd_traces', '_savers', '_string_summary_basic', '_time', '_time_default_to_arange', '_topology', '_unitcell_angles', '_unitcell_lengths', '_xyz', 'atom_slice', 'center_coordinates', 'image_molecules', 'join', 'load', 'make_molecules_whole', 'n_atoms', 'n_chains', 'n_frames', 'n_residues', 'openmm_boxes', 'openmm_positions', 'remove_solvent', 'restrict_atoms', 'save', 'save_amberrst7', 'save_binpos', 'save_dcd', 'save_dtr', 'save_gro', 'save_gsd', 'save_hdf5', 'save_lammpstrj', 'save_lh5', 'save_mdcrd', 'save_netcdf', 'save_netcdfr

In [215]:
print(dir(top))

['__class__', '__copy__', '__deepcopy__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_atoms', '_bonds', '_chains', '_numAtoms', '_numResidues', '_residues', '_standardBonds', '_string_summary_basic', '_unique_pairs', '_unique_pairs_equal', '_unique_pairs_mutually_exclusive', 'add_atom', 'add_bond', 'add_chain', 'add_residue', 'atom', 'atoms', 'atoms_by_name', 'bonds', 'chain', 'chains', 'copy', 'create_disulfide_bonds', 'create_standard_bonds', 'delete_atom_by_index', 'find_molecules', 'from_dataframe', 'from_openmm', 'guess_anchor_molecules', 'insert_atom', 'join', 'n_atoms', 'n_bonds', 'n_chains', 'n_residues', 'residue', 'residues', 'select', 'select_atom_indices', 'select_expression', 'select_pairs

In [208]:
print(dir(top.select_atom_indices))

['__call__', '__class__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__func__', '__ge__', '__get__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__self__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__']


In [242]:
print(dir(element))

['Element', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'actinium', 'aluminium', 'aluminum', 'americium', 'antimony', 'argon', 'arsenic', 'astatine', 'barium', 'berkelium', 'beryllium', 'bismuth', 'bohrium', 'boron', 'bromine', 'cadmium', 'calcium', 'californium', 'carbon', 'cerium', 'cesium', 'chlorine', 'chromium', 'cobalt', 'copper', 'curium', 'daltons', 'darmstadtium', 'deuterium', 'division', 'dubnium', 'dysprosium', 'einsteinium', 'erbium', 'europium', 'fermium', 'fluorine', 'francium', 'gadolinium', 'gallium', 'germanium', 'get_by_symbol', 'gold', 'hafnium', 'hassium', 'helium', 'holmium', 'hydrogen', 'indium', 'iodine', 'iridium', 'iron', 'is_quantity', 'krypton', 'lanthanum', 'lawrencium', 'lead', 'lithium', 'lutetium', 'magnesium', 'manganese', 'meitnerium', 'mendelevium', 'mercury', 'molybdenum', 'neodymium', 'neon', 'neptunium', 'nickel', 'niobium', 'nitrogen', 'nobelium', 'np', 'osmium', 'oxygen', 'palladium', '

In [15]:
# mdtraj.compute_distances(trajectory)
# mdtraj.compute_angles(trajectory, trajectory.angle_indices[0][0])
mdtraj.density(trajectory)

TypeError: unsupported operand type(s) for /: 'float' and 'NoneType'

In [16]:
new_traj.superpose(new_traj[0])

<mdtraj.Trajectory with 9970 frames, 166 atoms, 10 residues, without unitcells at 0x22203654e20>

In [303]:
bonds[:,:2]

array([[  4.,  21.],
       [ 21.,  22.],
       [  4.,   6.],
       [  4.,   5.],
       [  0.,   4.],
       [  6.,   9.],
       [  6.,   8.],
       [  6.,   7.],
       [ 10.,  12.],
       [  9.,  10.],
       [ 10.,  11.],
       [ 17.,  19.],
       [  9.,  17.],
       [ 17.,  18.],
       [ 12.,  14.],
       [ 12.,  13.],
       [ 14.,  19.],
       [ 19.,  20.],
       [ 14.,  15.],
       [  0.,   1.],
       [  0.,   2.],
       [  0.,   3.],
       [ 15.,  16.],
       [ 21.,  23.],
       [ 25.,  42.],
       [ 42.,  43.],
       [ 25.,  27.],
       [ 25.,  26.],
       [ 23.,  25.],
       [ 27.,  30.],
       [ 27.,  29.],
       [ 27.,  28.],
       [ 31.,  33.],
       [ 30.,  31.],
       [ 31.,  32.],
       [ 38.,  40.],
       [ 30.,  38.],
       [ 38.,  39.],
       [ 33.,  35.],
       [ 33.,  34.],
       [ 35.,  40.],
       [ 40.,  41.],
       [ 35.,  36.],
       [ 23.,  24.],
       [ 36.,  37.],
       [ 42.,  44.],
       [ 46.,  54.],
       [ 54.,

In [312]:
mdtraj.compute_neighbors(new_traj[0], 0.8, (12,))

[array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  13,
         14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,
         27,  28,  29,  30,  31,  32,  38,  39,  40,  42,  43, 159, 160,
        161, 162, 163, 164, 165])]

In [313]:
mdtraj.compute_distances(new_traj[1], bonds[:,:2])*10

array([[1.5269173 , 1.1913766 , 1.5188073 , 1.08      , 1.4968553 ,
        1.5186081 , 1.1110003 , 1.1110001 , 1.4227135 , 1.4076788 ,
        1.0799999 , 1.4338207 , 1.3958385 , 1.08      , 1.4191687 ,
        1.0800009 , 1.4219652 , 1.0799997 , 1.4153252 , 1.0399997 ,
        1.0400006 , 1.0399998 , 0.9600007 , 1.3798566 , 1.5873518 ,
        1.2417446 , 1.5869117 , 1.0799994 , 1.4020776 , 1.4385141 ,
        1.1110008 , 1.1109987 , 1.4322994 , 1.3976626 , 1.0800003 ,
        1.401638  , 1.3778858 , 1.0799999 , 1.4711999 , 1.0799997 ,
        1.3657473 , 1.0800004 , 1.3931876 , 0.99699986, 0.96000123,
        1.3840364 , 1.5182347 , 1.215678  , 1.5786619 , 1.0800005 ,
        1.4633417 , 1.5125253 , 1.1110004 , 1.1109998 , 1.2671782 ,
        1.2649863 , 0.9969996 , 1.3350534 , 1.5488732 , 1.2117527 ,
        1.5528104 , 1.0800003 , 1.4142158 , 1.4583786 , 1.1109997 ,
        1.1110001 , 1.5124332 , 1.1110005 , 1.1110005 , 1.4984531 ,
        1.1110004 , 1.1110001 , 1.3791627 , 1.49

In [6]:
atom_indices_list = []
for i in range(166):
    atom_indices_list.append(i)

In [ ]:
import nglview as nv
new = nv.show_mdtraj(new_traj)
# new = nv.show_mdtraj(traj)
# new = nv.show_mdtraj(trajectory)
# new.clear()
new.add_ball_and_stick('protein', opacity=0.5)
# new.add_surface('protein', opacity=0.05, color='residueindex')
new

In [39]:
coord = []
for i in range(len(trajectory)):
    coord.append(trajectory[i].xyz*10)

In [40]:
len(coord)

10000

In [20]:
import torch
from torch_geometric.data import Data

edge_index = torch.tensor([[0, 1, 1, 2],
                           [1, 0, 2, 1]], dtype=torch.long)
x = torch.tensor([[-1], [0], [1]], dtype=torch.float)

data = Data(x=x, edge_index=edge_index)

In [21]:
data

Data(x=[3, 1], edge_index=[2, 4])